# Notebook 01: Data Ingestion — UCI Online Retail Dataset

Loads the raw UCI Online Retail dataset into MySQL's staging table `raw_transactions` for downstream cleaning.

In [ ]:
# ── Imports ───────────────────────────────────────────
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

print('Libraries loaded ✓')

In [ ]:
# ── Database Connection Config ─────────────────────────
# Update DB_PASSWORD before running
DB_USER     = 'root'
DB_PASSWORD = 'your_password'   # ← Update DB_PASSWORD before running
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_NAME     = 'ecommerce_clv'

print('DB config set ✓')

In [ ]:
# ── Load Raw CSV ───────────────────────────────────────
DATA_PATH = Path('../data/raw/online_retail.csv')

df = pd.read_csv(
    DATA_PATH,
    encoding='ISO-8859-1',
    dtype={'CustomerID': str, 'InvoiceNo': str, 'StockCode': str}
)

print('Shape:', df.shape)
print()
print('Dtypes:')
print(df.dtypes)
print()
print('First 5 rows:')
df.head()

In [ ]:
# ── Create SQLAlchemy Engine ───────────────────────────
engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4',
    echo=False
)

print('Engine created ✓')

In [ ]:
# ── Load into MySQL staging table ─────────────────────
df.to_sql(
    'raw_transactions',
    engine,
    if_exists='replace',
    index=True,
    index_label='id',
    chunksize=5000,
    method='multi'
)

print(f'✅ Raw transactions loaded: {len(df):,} rows')

In [ ]:
# ── Verification Query ────────────────────────────────
verification = pd.read_sql(
    "SELECT COUNT(*) as row_count, "
    "COUNT(DISTINCT CustomerID) as customers, "
    "COUNT(DISTINCT InvoiceNo) as invoices "
    "FROM raw_transactions",
    engine
)

print('Verification result:')
print(verification)

## Summary

This notebook loaded the raw UCI Online Retail dataset (`online_retail.csv`) into MySQL.

- **Source file:** `data/raw/online_retail.csv`
- **Rows loaded:** 541,909 transactions (2010–2011 UK gift retailer data)
- **Staging table:** `raw_transactions` in `ecommerce_clv` database
- **Columns:** InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country
- **Encoding:** ISO-8859-1 (required for this dataset)
- **Load method:** `to_sql` with `chunksize=5000` and `method='multi'` for performance

**Next step:** Run `02_data_cleaning.ipynb` to clean and normalize into relational tables.